In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

import vectorbt as vbt

# Prepare data
start = '2019-01-01 UTC'  # crypto is in UTC
end = '2020-01-01 UTC'
btc_price = vbt.YFData.download('BTC-USD', start=start, end=end).get('Close')

btc_price


Date
2019-01-01 00:00:00+00:00    3843.520020
2019-01-02 00:00:00+00:00    3943.409424
2019-01-03 00:00:00+00:00    3836.741211
2019-01-04 00:00:00+00:00    3857.717529
2019-01-05 00:00:00+00:00    3845.194580
                                ...     
2019-12-27 00:00:00+00:00    7290.088379
2019-12-28 00:00:00+00:00    7317.990234
2019-12-29 00:00:00+00:00    7422.652832
2019-12-30 00:00:00+00:00    7292.995117
2019-12-31 00:00:00+00:00    7193.599121
Freq: D, Name: Close, Length: 365, dtype: float64

In [2]:
fast_ma = vbt.MA.run(btc_price, 10, short_name='fast')
slow_ma = vbt.MA.run(btc_price, 20, short_name='slow')

entries = fast_ma.ma_crossed_above(slow_ma)
entries



Date
2019-01-01 00:00:00+00:00    False
2019-01-02 00:00:00+00:00    False
2019-01-03 00:00:00+00:00    False
2019-01-04 00:00:00+00:00    False
2019-01-05 00:00:00+00:00    False
                             ...  
2019-12-27 00:00:00+00:00     True
2019-12-28 00:00:00+00:00    False
2019-12-29 00:00:00+00:00    False
2019-12-30 00:00:00+00:00    False
2019-12-31 00:00:00+00:00    False
Freq: D, Length: 365, dtype: bool

In [3]:
exits = fast_ma.ma_crossed_below(slow_ma)
exits

Date
2019-01-01 00:00:00+00:00    False
2019-01-02 00:00:00+00:00    False
2019-01-03 00:00:00+00:00    False
2019-01-04 00:00:00+00:00    False
2019-01-05 00:00:00+00:00    False
                             ...  
2019-12-27 00:00:00+00:00    False
2019-12-28 00:00:00+00:00    False
2019-12-29 00:00:00+00:00    False
2019-12-30 00:00:00+00:00    False
2019-12-31 00:00:00+00:00    False
Freq: D, Length: 365, dtype: bool

In [4]:
pf = vbt.Portfolio.from_signals(btc_price, entries, exits)
pf.total_return()

np.float64(0.6351860771192923)

In [5]:
# Multiple strategy instances: (10, 30) and (20, 30)
fast_ma = vbt.MA.run(btc_price, [10, 20], short_name='fast')
slow_ma = vbt.MA.run(btc_price, [30, 30], short_name='slow')

entries = fast_ma.ma_crossed_above(slow_ma)
entries


fast_window,10,20
slow_window,30,30
Date,,
2019-01-01 00:00:00+00:00,False,False
2019-01-02 00:00:00+00:00,False,False
2019-01-03 00:00:00+00:00,False,False
2019-01-04 00:00:00+00:00,False,False
2019-01-05 00:00:00+00:00,False,False
...,...,...
2019-12-27 00:00:00+00:00,False,False
2019-12-28 00:00:00+00:00,False,False


In [6]:
exits = fast_ma.ma_crossed_below(slow_ma)
exits

fast_window,10,20
slow_window,30,30
Date,,
2019-01-01 00:00:00+00:00,False,False
2019-01-02 00:00:00+00:00,False,False
2019-01-03 00:00:00+00:00,False,False
2019-01-04 00:00:00+00:00,False,False
2019-01-05 00:00:00+00:00,False,False
...,...,...
2019-12-27 00:00:00+00:00,False,False
2019-12-28 00:00:00+00:00,False,False


In [7]:
pf = vbt.Portfolio.from_signals(btc_price, entries, exits)
pf.total_return()


fast_window  slow_window
10           30             0.847151
20           30             0.543411
Name: total_return, dtype: float64

In [8]:
# Multiple strategy instances and instruments
eth_price = vbt.YFData.download('ETH-USD', start=start, end=end).get('Close')
comb_price = btc_price.vbt.concat(eth_price,
    keys=pd.Index(['BTC', 'ETH'], name='symbol'))
comb_price.vbt.drop_levels(-1, inplace=True)
comb_price

symbol,BTC,ETH
Date,,
2019-01-01 00:00:00+00:00,3843.520020,140.819412
2019-01-02 00:00:00+00:00,3943.409424,155.047684
2019-01-03 00:00:00+00:00,3836.741211,149.135010
2019-01-04 00:00:00+00:00,3857.717529,154.581940
2019-01-05 00:00:00+00:00,3845.194580,155.638596
...,...,...
2019-12-27 00:00:00+00:00,7290.088379,127.214607
2019-12-28 00:00:00+00:00,7317.990234,128.322708
2019-12-29 00:00:00+00:00,7422.652832,134.757980


In [9]:
fast_ma = vbt.MA.run(comb_price, [10, 20], short_name='fast')
slow_ma = vbt.MA.run(comb_price, [30, 30], short_name='slow')

entries = fast_ma.ma_crossed_above(slow_ma)
entries

fast_window                   10            20       
slow_window                   30            30       
symbol                       BTC    ETH    BTC    ETH
Date                                                 
2019-01-01 00:00:00+00:00  False  False  False  False
2019-01-02 00:00:00+00:00  False  False  False  False
2019-01-03 00:00:00+00:00  False  False  False  False
2019-01-04 00:00:00+00:00  False  False  False  False
2019-01-05 00:00:00+00:00  False  False  False  False
...                          ...    ...    ...    ...
2019-12-27 00:00:00+00:00  False  False  False  False
2019-12-28 00:00:00+00:00  False  False  False  False
2019-12-29 00:00:00+00:00   True  False  False  False
2019-12-30 00:00:00+00:00  False  False  False  False
2019-12-31 00:00:00+00:00  False  False  False  False

[365 rows x 4 columns]

In [10]:
exits = fast_ma.ma_crossed_below(slow_ma)
exits


fast_window                   10            20       
slow_window                   30            30       
symbol                       BTC    ETH    BTC    ETH
Date                                                 
2019-01-01 00:00:00+00:00  False  False  False  False
2019-01-02 00:00:00+00:00  False  False  False  False
2019-01-03 00:00:00+00:00  False  False  False  False
2019-01-04 00:00:00+00:00  False  False  False  False
2019-01-05 00:00:00+00:00  False  False  False  False
...                          ...    ...    ...    ...
2019-12-27 00:00:00+00:00  False  False  False  False
2019-12-28 00:00:00+00:00  False  False  False  False
2019-12-29 00:00:00+00:00  False  False  False  False
2019-12-30 00:00:00+00:00  False  False  False  False
2019-12-31 00:00:00+00:00  False  False  False  False

[365 rows x 4 columns]

In [11]:
pf = vbt.Portfolio.from_signals(comb_price, entries, exits)
pf.total_return()

fast_window  slow_window  symbol
10           30           BTC       0.847151
                          ETH       0.244204
20           30           BTC       0.543411
                          ETH      -0.319102
Name: total_return, dtype: float64

In [12]:
mean_return = pf.total_return().groupby('symbol').mean()
mean_return.vbt.barplot(xaxis_title='Symbol', yaxis_title='Mean total return')

FigureWidget({
    'data': [{'name': 'total_return',
              'showlegend': True,
              'type': 'bar',
              'uid': '4919602d-a336-407c-9769-b85706bd6e4d',
              'x': array(['BTC', 'ETH'], dtype=object),
              'y': {'bdata': 'diLf074/5j8on9lMjSyjvw==', 'dtype': 'f8'}}],
    'layout': {'height': 350,
               'legend': {'orientation': 'h',
                          'traceorder': 'normal',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'margin': {'b': 30, 'l': 30, 'r': 30, 't': 30},
               'template': '...',
               'width': 700,
               'xaxis': {'title': {'text': 'Symbol'}},
               'yaxis': {'title': {'text': 'Mean total return'}}}
})

In [15]:
# Multiple strategy instances, instruments, and time periods
mult_comb_price, _ = comb_price.vbt.range_split(n=2)
mult_comb_price

split_idx             0                         1            
symbol              BTC         ETH           BTC         ETH
0           3843.520020  140.819412  11961.269531  303.099976
1           3943.409424  155.047684  11215.437500  284.523224
2           3836.741211  149.135010  10978.459961  287.997528
3           3857.717529  154.581940  11208.550781  287.547119
4           3845.194580  155.638596  11450.846680  305.700562
..                  ...         ...           ...         ...
177        11182.806641  294.267639   7290.088379  127.214607
178        12407.332031  311.226105   7317.990234  128.322708
179        11959.371094  320.058899   7422.652832  134.757980
180        10817.155273  290.695984   7292.995117  132.633484
181        10583.134766  293.641113   7193.599121  129.610855

[182 rows x 4 columns]

In [19]:
fast_ma = vbt.MA.run(mult_comb_price, [10, 20], short_name='fast')
slow_ma = vbt.MA.run(mult_comb_price, [30, 30], short_name='slow')

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

pf = vbt.Portfolio.from_signals(mult_comb_price, entries, exits, freq='1D')
pf.total_return()

fast_window  slow_window  split_idx  symbol
10           30           0          BTC       1.579002
                                     ETH       0.960437
                          1          BTC      -0.289369
                                     ETH      -0.308387
20           30           0          BTC       1.666387
                                     ETH       0.352693
                          1          BTC      -0.418280
                                     ETH      -0.257947
Name: total_return, dtype: float64

In [20]:
mean_return = pf.total_return().groupby(['split_idx', 'symbol']).mean()
mean_return.unstack(level=-1).vbt.barplot(
    xaxis_title='Split index',
    yaxis_title='Mean total return',
    legend_title_text='Symbol')

FigureWidget({
    'data': [{'name': 'BTC',
              'showlegend': True,
              'type': 'bar',
              'uid': 'f403df30-67d4-45e2-ac16-3f1212c70557',
              'x': {'bdata': 'AAE=', 'dtype': 'i1'},
              'y': {'bdata': 'ZqzMnY72+T+Kd+2TD6XWvw==', 'dtype': 'f8'}},
             {'name': 'ETH',
              'showlegend': True,
              'type': 'bar',
              'uid': '93d6d5e1-7049-499c-b6b7-a6632b503098',
              'x': {'bdata': 'AAE=', 'dtype': 'i1'},
              'y': {'bdata': 'f4SG3JQC5T9BSQRgaR/Svw==', 'dtype': 'f8'}}],
    'layout': {'height': 350,
               'legend': {'orientation': 'h',
                          'title': {'text': 'Symbol'},
                          'traceorder': 'normal',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'margin': {'b': 30, 'l': 30, 'r': 30, 't': 30},
               